In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_theme(style='whitegrid')

print('Libraries loaded successfully.')

Libraries loaded successfully.


In [ ]:
df = pd.read_csv('titanic.csv')
print(f'Dataset shape: {df.shape}')
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')

In [ ]:
print('=== First 5 Rows of Dataset ===')
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    df['Age'], df['Fare'],
    c=df['Survived'], cmap='coolwarm',
    alpha=0.6, edgecolors='k', linewidths=0.4, s=60
)
legend = ax.legend(*scatter.legend_elements(), title='Survived', labels=['Did Not Survive', 'Survived'])
ax.add_artist(legend)
ax.set_title('Scatter Plot: Age vs. Fare (colored by Survival)', fontsize=14, fontweight='bold')
ax.set_xlabel('Age')
ax.set_ylabel('Fare ($)')
plt.tight_layout()
plt.show()

print("""
Insight: Passengers who paid higher fares (1st class) cluster at higher ages and show
a higher survival rate (blue). Low-fare passengers are concentrated at lower ages and
are predominantly non-survivors (red).
""")

In [ ]:
survival_by_class = df.groupby(['Pclass', 'Survived']).size().unstack(fill_value=0)
survival_by_class.columns = ['Did Not Survive', 'Survived']
survival_by_class.index = ['1st Class', '2nd Class', '3rd Class']

ax = survival_by_class.plot(kind='bar', color=['#e74c3c', '#2ecc71'], edgecolor='black', figsize=(9, 6))
ax.set_title('Bar Chart: Survival Count by Passenger Class', fontsize=14, fontweight='bold')
ax.set_xlabel('Passenger Class')
ax.set_ylabel('Number of Passengers')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend()
plt.tight_layout()
plt.show()

print("""
Insight: 1st class passengers had the highest survival rate relative to class size.
3rd class passengers experienced the most deaths, reflecting socioeconomic disparity.
""")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(df['Age'].dropna(), bins=30, color='steelblue', edgecolor='white', linewidth=0.7)
ax.axvline(df['Age'].mean(), color='red', linestyle='--', linewidth=1.5, label=f"Mean: {df['Age'].mean():.1f}")
ax.axvline(df['Age'].median(), color='orange', linestyle='--', linewidth=1.5, label=f"Median: {df['Age'].median():.1f}")
ax.set_title('Histogram: Age Distribution of Passengers', fontsize=14, fontweight='bold')
ax.set_xlabel('Age')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

print("""
Insight: Age is right-skewed; most passengers were 20–40 years old.
Mean (29.7) slightly exceeds median (28.0). A small spike exists for children (0–5).
""")

In [ ]:
sex_counts = df['Sex'].value_counts()

fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(sex_counts, labels=['Male', 'Female'], autopct='%1.1f%%',
       colors=['#5b9bd5', '#f4a261'], startangle=140,
       wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax.set_title('Pie Chart: Passenger Gender Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("""
Insight: ~64.8% male, ~35.2% female. Despite being the minority, females had a significantly
higher survival rate due to the 'women and children first' evacuation policy.
""")

In [ ]:
df_sorted = df.sort_values('PassengerId').reset_index(drop=True)
df_sorted['CumulativeSurvived'] = df_sorted['Survived'].cumsum()

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(df_sorted['PassengerId'], df_sorted['CumulativeSurvived'],
        color='teal', linewidth=2, label='Cumulative Survivors')
ax.set_title('Line Plot: Cumulative Survivors over Passenger ID Sequence', fontsize=14, fontweight='bold')
ax.set_xlabel('Passenger ID')
ax.set_ylabel('Cumulative Survivors')
ax.legend()
plt.tight_layout()
plt.show()

print("""
Insight: Cumulative survivor count grows roughly linearly — survivors are distributed
throughout the manifest. Final total: ~342 survivors out of 891 (~38.4% survival rate).
""")

In [ ]:
print('=== Missing Values BEFORE Preprocessing ===')
missing_before = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_report = pd.DataFrame({'Missing Count': missing_before, 'Missing %': missing_pct})
print(missing_report[missing_report['Missing Count'] > 0])
print(f'\nDataset shape before: {df.shape}')
df.head(5)

In [ ]:
df_clean = df.copy()

# Age: fill with median
age_median = df_clean['Age'].median()
df_clean['Age'] = df_clean['Age'].fillna(age_median)
print(f'Age: filled {df["Age"].isnull().sum()} missing values with median ({age_median})')

# Embarked: fill with mode
embarked_mode = df_clean['Embarked'].mode()[0]
df_clean['Embarked'] = df_clean['Embarked'].fillna(embarked_mode)
print(f'Embarked: filled {df["Embarked"].isnull().sum()} missing values with mode ("{embarked_mode}")')

# Cabin: drop column (>77% missing)
df_clean.drop(columns=['Cabin'], inplace=True)
print('Cabin: column dropped (77%+ missing — not imputable)')

In [ ]:
print('=== Missing Values AFTER Preprocessing ===')
print(df_clean.isnull().sum())
print(f'\nDataset shape after: {df_clean.shape}')
df_clean.head(5)